## Backend-Plugin mit Ping-Endpunkt

Das Backend-Plugin wird mit dem offiziellen `backend-plugin`-Template erzeugt.

Es stellt den minimalen Endpunkt `GET /api/ping/ping` bereit.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn new --select backend-plugin --option pluginId=ping

Der Code erstellt einen Express-Router für das Backend-Plugin, damit das Plugin einen eigenen HTTP-Endpunkt bereitstellen kann. 

Ein Aufruf von `/ping` liefert die JSON-Antwort `{"message":"pong"}` und dient als einfacher Funktionstest.


In [ ]:
%%bash
cat > plugins/ping-backend/src/router.ts <<'EOF'
import { Router } from 'express';

export function createRouter(): Router {
  const router = Router();

  router.get('/ping', (_request, response) => {
    response.json({ message: 'pong' });
  });

  return router;
}
EOF


Der Code registriert das Ping-Plugin im Backstage-Backend, bindet den zuvor definierten Router ein und erlaubt den Zugriff auf `/ping` ohne Anmeldung. 

Die Datei `index.ts` exportiert das Plugin als Standardeinstiegspunkt, damit Backstage es laden kann.


In [ ]:
%%bash
cat > plugins/ping-backend/src/plugin.ts <<'EOF'
import {
  coreServices,
  createBackendPlugin,
} from '@backstage/backend-plugin-api';
import { createRouter } from './router';

export const pingPlugin = createBackendPlugin({
  pluginId: 'ping',
  register(env) {
    env.registerInit({
      deps: {
        httpRouter: coreServices.httpRouter,
      },
      async init({ httpRouter }) {
        httpRouter.use(createRouter());
        httpRouter.addAuthPolicy({
          path: '/ping',
          allow: 'unauthenticated',
        });
      },
    });
  },
});
EOF

cat > plugins/ping-backend/src/index.ts <<'EOF'
export { pingPlugin as default } from './plugin';
EOF


Das CLI-Template trägt das Backend-Plugin normalerweise bereits in
`packages/backend/src/index.ts` ein. Die folgende Ausgabe dient nur zur Kontrolle.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
grep "plugin-ping-backend" packages/backend/src/index.ts

**Backstage starten:** Backstage wird mit dem neuen Plugin gestartet.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME=$(cat ~/data/server-ip)
export BACKSTAGE_NAME="Backstage Produktion"
export BACKSTAGE_PORT="3002"

echo "http://$(cat ~/data/server-ip):${BACKSTAGE_PORT}"
source ~/.nvm/nvm.sh
cd ~/mybackstage
yarn start --config ~/mybackstage/app-config.yaml --config ~/mybackstage/app-config.production.yaml


**Testen**

New Launcher `+`-> Terminal

Im Terminal folgenden Befehle ausführen (kopieren mit Ctrl+c und einfügen mittels Ctrl+v)

    curl http://localhost:7007/api/ping/ping